# Data Exploration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math

In [2]:
import sys
from pathlib import Path

ruta_proyecto = Path().resolve().parent / "my_project"

if str(ruta_proyecto) not in sys.path:
    sys.path.insert(1, str(ruta_proyecto))

from fileManager import FileManager

In [3]:
def inspect(df):
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(f"Data types:\n{df.dtypes}")
    print()

In [4]:
RAW_FILE = "../data/raw/automobile_dataset"
PROCESSED_TRAIN_FILE = "../data/processed/automobile_dataset"
PROCESSED_TEST_FILE = "../data/processed/automobile_test"
EXPLORATION_IMAGES_FOLDER = "../reports/figures/exploration/"

In [5]:
fM = FileManager()

In [6]:
fM.set_format("csv")
df = fM.read(RAW_FILE)

display(df)

,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
0,Mercedes-Benz,GLE,2024,Petrol,Automatic,2.3,100,186.0,196.0,1,0.0,NaN,Silver,SUV,AWD,30.0,IL,64140
1,Hyundai,Tucson,2008,Petrol,Automatic,2.3,387035,189.0,190.0,4,0.0,NaN,Blue,SUV,FWD,35.0,FL,500
2,Volkswagen,Golf,2021,Hybrid,NaN,1.9,46054,158.0,153.0,1,0.0,Partial Service,Gray,Hatchback,AWD,43.0,NY,16429
3,Chevrolet,Tahoe,2005,Petrol,NaN,2.2,141302,169.0,165.0,5,1.0,No Service,Blue,SUV,AWD,NaN,FL,2199
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5495,Volkswagen,Tiguan,2022,Petrol,Automatic,NaN,29506,161.0,NaN,1,0.0,Partial Service,Black,SUV,AWD,33.0,GA,20333
5496,Nissan,Pathfinder,2006,Petrol,Automatic,3.1,287332,243.0,227.0,5,1.0,No Service,Gray,SUV,AWD,NaN,NY,500
5497,BMW,X3,2013,Petrol,Automatic,2.2,95961,171.0,152.0,2,1.0,No Service,Blue,SUV,AWD,32.0,MI,9882
5498,Volkswagen,Golf,2020,Hybrid,Automatic,1.5,56643,135.0,125.0,2,1.0,No Service,White,Hatchback,FWD,57.0,NY,10474


In [7]:
inspect(df)

Shape: (5500, 18)
Columns: ['Make', 'Model', 'Year', 'Fuel_Type', 'Transmission', 'Engine_Size', 'Mileage', 'Horsepower', 'Torque', 'Owners', 'Accident_History', 'Service_History', 'Color', 'Body_Type', 'Drivetrain', 'Fuel_Efficiency', 'Location', 'Selling_Price']
Data types:
Make                 object
Model                object
Year                  int64
Fuel_Type            object
Transmission         object
Engine_Size         float64
Mileage               int64
Horsepower          float64
Torque              float64
Owners                int64
Accident_History    float64
Service_History      object
Color                object
Body_Type            object
Drivetrain           object
Fuel_Efficiency     float64
Location             object
Selling_Price         int64
dtype: object



## Memory optimization

In [8]:
# Initial memory usage
initial_size = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Initial memory usage: {initial_size:.2f} MB")

# Optimize data types
for col in list(df.columns):
    if df.dtypes[col] == "int64":
        df[col] = pd.to_numeric(df[col], downcast='integer')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after downcasting to integer: {size:.2f} MB")
    if df.dtypes[col] == "float64":
        df[col] = pd.to_numeric(df[col], downcast='float')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after downcasting to float: {size:.2f} MB")
    if df.dtypes[col] == "str" or df.dtypes[col] == "object":
        df[col] = df[col].astype('category')
        size = df.memory_usage(deep=True).sum() / (1024 ** 2)
        #print(f"Memory usage after changing to categorical column: {size:.2f} MB")

# Optimized memory usage
final_size = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Final memory usage: {final_size:.2f} MB")
print(f"Reduction of {(1 - final_size / initial_size) * 100:.2f}%")

Initial memory usage: 2.90 MB
Final memory usage: 0.22 MB
Reduction of 92.51%


In [9]:
inspect(df)

Shape: (5500, 18)
Columns: ['Make', 'Model', 'Year', 'Fuel_Type', 'Transmission', 'Engine_Size', 'Mileage', 'Horsepower', 'Torque', 'Owners', 'Accident_History', 'Service_History', 'Color', 'Body_Type', 'Drivetrain', 'Fuel_Efficiency', 'Location', 'Selling_Price']
Data types:
Make                category
Model               category
Year                   int16
Fuel_Type           category
Transmission        category
Engine_Size          float32
Mileage                int32
Horsepower           float32
Torque               float32
Owners                  int8
Accident_History     float32
Service_History     category
Color               category
Body_Type           category
Drivetrain          category
Fuel_Efficiency      float32
Location            category
Selling_Price          int32
dtype: object



## Missing values 

In [10]:
print("Missing values:")
print(df.isnull().sum() > 0)
print()

print("Fill missing numerical values:")
dictionary = {}

# Numerical Fixes:
dfk = df.isnull().sum() > 0
for col in list(df.columns):
    if dfk[col] and df.dtypes[col] != "category":
        dictionary[col] = dfk[col].mean()
df.fillna(dictionary)

print("Drop rows with any missing categorical values:")
print("Categorical Fixes:")
df = df.dropna()
display(df)

# Missing values:
print(df.isnull().sum() > 0)
print()

Missing values:
Make                False
Model               False
Year                False
Fuel_Type           False
Transmission         True
Engine_Size          True
Mileage             False
Horsepower           True
Torque               True
Owners              False
Accident_History     True
Service_History      True
Color                True
Body_Type           False
Drivetrain          False
Fuel_Efficiency      True
Location             True
Selling_Price       False
dtype: bool

Fill missing numerical values:
Drop rows with any missing categorical values:
Categorical Fixes:


,Make,Model,Year,Fuel_Type,Transmission,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Color,Body_Type,Drivetrain,Fuel_Efficiency,Location,Selling_Price
4,Toyota,Camry,2022,Petrol,Automatic,1.9,32813,149.0,141.0,1,0.0,Full Service,Brown,Sedan,FWD,38.0,CA,21792
7,Mercedes-Benz,C-Class,2022,Petrol,Automatic,2.3,32133,180.0,162.0,1,0.0,Partial Service,Green,Sedan,FWD,35.0,NY,31416
15,Mercedes-Benz,C-Class,2010,Petrol,Manual,2.0,17138,169.0,160.0,3,0.0,Partial Service,Blue,Sedan,FWD,27.0,NC,11728
17,Chevrolet,Tahoe,2011,Petrol,Automatic,3.3,264441,258.0,252.0,4,0.0,Partial Service,Gray,SUV,AWD,22.0,TX,3395
21,Audi,Q7,2016,Petrol,Automatic,3.1,111537,243.0,219.0,2,0.0,Partial Service,Blue,SUV,AWD,21.0,FL,23314
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5478,Audi,A6,2010,Petrol,Manual,2.1,155818,169.0,176.0,3,0.0,Full Service,Red,Sedan,FWD,41.0,FL,9714
5479,Honda,Accord,2023,Petrol,Automatic,2.4,100,199.0,192.0,1,0.0,Partial Service,Green,Sedan,FWD,33.0,GA,23937
5491,Nissan,Pathfinder,2020,Petrol,Automatic,2.7,79498,212.0,204.0,1,0.0,Partial Service,Gray,SUV,AWD,24.0,OH,18345
5497,BMW,X3,2013,Petrol,Automatic,2.2,95961,171.0,152.0,2,1.0,No Service,Blue,SUV,AWD,32.0,MI,9882


Make                False
Model               False
Year                False
Fuel_Type           False
Transmission        False
Engine_Size         False
Mileage             False
Horsepower          False
Torque              False
Owners              False
Accident_History    False
Service_History     False
Color               False
Body_Type           False
Drivetrain          False
Fuel_Efficiency     False
Location            False
Selling_Price       False
dtype: bool



## Exploration 

#### 1. Car Model Count 

In [11]:
plt.figure(figsize=(10, 5))
car_counts = df["Make"].value_counts()
plt.bar(car_counts.index, car_counts.values , color='royalblue')
plt.xlabel("Car Model")
plt.ylabel("Count")
plt.xticks(rotation=90) 
plt.title("Automobile Dataset: Car Model Count")
plt.tight_layout()
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}car_model_count.png", dpi=300)

plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
# plt.show() # IF WE WANT TO SHOW THE IMAGE

Con este gráfico vemos que el conjunto de datos está balanceado, hay un numero parecido de coches de cada marca.

#### 2. Heatmap

In [12]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(7, 5))

sns.heatmap( corr, annot=True, cmap='coolwarm' , fmt=".2f")

plt.tight_layout()
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}heatmap.png", dpi=300)

plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
# plt.show() # IF WE WANT TO SHOW THE IMAGE

Correlaciones altas:
- Horsepower with Torque (0.98)
- Selling price está relacionado con Year positivamente (0.81) (más nuevo -> más caro) 
- Selling_Price está negativamente correlacionado con Mileage (-0.69) y con Owners (-0.68). Es decir, conforme sube el Mileage, el precio Selling_price disminuirá.
- Engine_size está negativamente relacionado con Fuel_Efficiency (-0.74). Es decir, conforme mayor sera el tamaño del Engine, disminuye la fuel efficiency.
- Engine_size está relacionado con Horsepower (0.67) y con Torque (0.56).

#### 3. Pruebas de Scatterplot

In [13]:
makes = df["Make"].unique()
n = len(makes)

cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = axes.flatten()

for ax, make in zip(axes, makes):
    df_make = df[df["Make"] == make]

    sns.scatterplot(
        data=df_make,
        x="Selling_Price",
        y="Mileage",
        color="skyblue",
        ax=ax
    )

    ax.set_title(make)

# Eliminar ejes vacíos
for ax in axes[len(makes):]:
    fig.delaxes(ax)

plt.tight_layout()
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}scatterplots.png", dpi=300)

plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
# plt.show() # IF WE WANT TO SHOW THE IMAGE

#### 4. Boxplot by Category

The most expensive cars are: Audi, BMW, Mercedes-Benz. They also have the most expensive automobiles.

In [14]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Make", y="Selling_Price", palette="Set2")
plt.xticks(rotation=90) 
plt.title("Selling Price vs Car ")
plt.tight_layout()

plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}selling_vs_car.png", dpi=300)

plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
# plt.show() # IF WE WANT TO SHOW THE IMAGE

/tmp/ipykernel_245592/2576894321.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="Make", y="Selling_Price", palette="Set2")


#### 5. Seaborn Pairplot Experiments

In [15]:
sns.set_theme(style="white")
sns.pairplot(df, hue="Selling_Price", height=2.5)
plt.suptitle("Iris dataset: Pairplot", y=1.02)
plt.savefig(f"{EXPLORATION_IMAGES_FOLDER}pairplots.png", dpi=300)

plt.close() # IF WE DONT WANT TO SHOW THE IMAGE.
#plt.show() # IF WE WANT TO SHOW THE IMAGE

## Categorical to numerical

In [16]:
print(df["Make"].nunique())
print(df["Color"].nunique())
print(df["Model"].nunique())
print(df["Service_History"].nunique())
print(df["Body_Type"].nunique())
print(df["Drivetrain"].nunique())
print(df["Location"].nunique())

10
8
40
3
5
4
10


In [17]:
# Ordinal for service (from no service to full service, hierarchy is preserved)
service_mapping = {
    "No Service": 0,
    "Partial Service": 1,
    "Full Service": 2
}

df["Service_History"] = df["Service_History"].map(service_mapping)


# OHE for the other categorical columns
categorical_columns = [
    "Make",
    "Model",
    "Fuel_Type",
    "Transmission",
    "Color",
    "Body_Type",
    "Drivetrain",
    "Location"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    dtype=int
)

In [18]:
print(df.head())
print(df.dtypes)
print(df.shape)
print(df.columns)

    Year  Engine_Size  Mileage  Horsepower  Torque  Owners  Accident_History  \
4   2022          1.9    32813       149.0   141.0       1               0.0   
7   2022          2.3    32133       180.0   162.0       1               0.0   
15  2010          2.0    17138       169.0   160.0       3               0.0   
17  2011          3.3   264441       258.0   252.0       4               0.0   
21  2016          3.1   111537       243.0   219.0       2               0.0   

   Service_History  Fuel_Efficiency  Selling_Price  ...  Location_CA  \
4                2             38.0          21792  ...            1   
7                1             35.0          31416  ...            0   
15               1             27.0          11728  ...            0   
17               1             22.0           3395  ...            0   
21               1             21.0          23314  ...            0   

    Location_FL  Location_GA  Location_IL  Location_MI  Location_NC  \
4             0

## Save processed data

In [19]:
from sklearn.model_selection import train_test_split

# 80% training data
# 20% testing data
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=22
)

In [20]:
fM.set_format("parquet")
fM.write(train_df, PROCESSED_TRAIN_FILE)
fM.write(test_df, PROCESSED_TEST_FILE)

## Check it was correctly saved

In [21]:
fM.set_format("parquet")
fdf = fM.read(PROCESSED_TRAIN_FILE)

display(fdf)
inspect(fdf)

,Year,Engine_Size,Mileage,Horsepower,Torque,Owners,Accident_History,Service_History,Fuel_Efficiency,Selling_Price,...,Location_CA,Location_FL,Location_GA,Location_IL,Location_MI,Location_NC,Location_NY,Location_OH,Location_PA,Location_TX
633,2008,2.0,20347,167.0,151.0,5,1.0,1,35.0,3428,...,0,1,0,0,0,0,0,0,0,0
1028,2016,2.2,64385,162.0,167.0,4,1.0,1,29.0,5850,...,0,0,0,0,1,0,0,0,0,0
4550,2013,2.5,158156,187.0,172.0,3,0.0,1,22.0,1281,...,0,1,0,0,0,0,0,0,0,0
1056,2013,2.5,131422,209.0,189.0,3,1.0,1,27.0,7586,...,0,0,0,1,0,0,0,0,0,0
1063,2008,3.5,270715,263.0,239.0,4,0.0,0,19.0,500,...,0,0,0,1,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1461,2005,3.3,377675,250.0,231.0,3,0.0,1,24.0,500,...,0,0,0,0,0,0,0,0,1,0
4038,2017,1.6,87365,122.0,103.0,1,0.0,0,43.0,15518,...,0,0,0,1,0,0,0,0,0,0
3372,2023,3.2,3623,243.0,246.0,1,0.0,1,23.0,24300,...,0,0,0,0,0,0,0,0,0,1
497,2011,3.3,229650,238.0,223.0,3,1.0,1,24.0,7157,...,0,0,0,0,0,0,0,1,0,0


Shape: (1063, 93)
Columns: ['Year', 'Engine_Size', 'Mileage', 'Horsepower', 'Torque', 'Owners', 'Accident_History', 'Service_History', 'Fuel_Efficiency', 'Selling_Price', 'Make_Audi', 'Make_BMW', 'Make_Chevrolet', 'Make_Ford', 'Make_Honda', 'Make_Hyundai', 'Make_Mercedes-Benz', 'Make_Nissan', 'Make_Toyota', 'Make_Volkswagen', 'Model_3 Series', 'Model_5 Series', 'Model_A4', 'Model_A6', 'Model_Accord', 'Model_Altima', 'Model_Atlas', 'Model_C-Class', 'Model_CR-V', 'Model_Camry', 'Model_Civic', 'Model_Corolla', 'Model_E-Class', 'Model_Elantra', 'Model_Equinox', 'Model_Escape', 'Model_Explorer', 'Model_F-150', 'Model_GLC', 'Model_GLE', 'Model_Golf', 'Model_Highlander', 'Model_Malibu', 'Model_Mustang', 'Model_Passat', 'Model_Pathfinder', 'Model_Pilot', 'Model_Q5', 'Model_Q7', 'Model_RAV4', 'Model_Rogue', 'Model_Santa Fe', 'Model_Sentra', 'Model_Silverado', 'Model_Sonata', 'Model_Tahoe', 'Model_Tiguan', 'Model_Tucson', 'Model_X3', 'Model_X5', 'Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Ty